# Thesis-Grade 1D Conditional Diffusion for ECG Refinement (v5 - Final Linear Layout)

### Layout of this Notebook:
1. **Definitions:** Loads the Dataset, Model, Diffusion, and Metrics.
2. **Training (Scrollable Log):** Trains the model and prints the metrics line-by-line for every epoch.
3. **Visualizations:** Plots the Learning Curves and Signal Artifacts *after* training is complete.
4. **Thesis Comparison Table:** Runs a final evaluation batch to print a direct "Baseline vs Model" table.
5. **Thesis Decipher Guide:** A text cell explaining exactly what your results mean and how to compare them to other papers.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import welch
from scipy.stats import ks_2samp
import math
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

PATH_CLEAN = '/content/cleannew2.npy'
PATH_NOISY = '/content/noisynew2.npy'

In [ ]:
class RealECGDataset(Dataset):
    def __init__(self, clean_npy_path, noisy_npy_path):
        if not os.path.exists(clean_npy_path) or not os.path.exists(noisy_npy_path):
            raise FileNotFoundError(f"Please upload {clean_npy_path} and {noisy_npy_path} to Colab.")
            
        self.clean_data = np.load(clean_npy_path).astype(np.float32)
        self.noisy_data = np.load(noisy_npy_path).astype(np.float32)
        
        if len(self.clean_data.shape) == 2:
            self.clean_data = np.expand_dims(self.clean_data, axis=1)
            self.noisy_data = np.expand_dims(self.noisy_data, axis=1)
            
        print(f"Dataset loaded! Shape: {self.clean_data.shape}")

    def __len__(self):
        return len(self.clean_data)

    def __getitem__(self, idx):
        clean = torch.tensor(self.clean_data[idx])
        noisy = torch.tensor(self.noisy_data[idx])
        return clean, noisy

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class Block1D(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels)
        )
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU()
        )

    def forward(self, x, t):
        time_emb = self.mlp(t).unsqueeze(-1)
        return self.conv(x) + time_emb

class ConditionalUNet1D(nn.Module):
    def __init__(self, channels=1, time_emb_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU()
        )
        
        self.init_conv = nn.Conv1d(channels * 2, 64, kernel_size=7, padding=3)
        
        self.downs = nn.ModuleList([
            Block1D(64, 128, time_emb_dim),
            Block1D(128, 256, time_emb_dim)
        ])
        
        self.pool = nn.MaxPool1d(2)
        self.mid = Block1D(256, 256, time_emb_dim)
        
        self.ups = nn.ModuleList([
            Block1D(256 + 256, 128, time_emb_dim), 
            Block1D(128 + 128, 64, time_emb_dim)   
        ])
        
        self.up_sample = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)
        self.final_conv = nn.Conv1d(64, channels, kernel_size=3, padding=1)

    def forward(self, x, t, cond):
        t = self.time_mlp(t)
        x = torch.cat([x, cond], dim=1) 
        x = self.init_conv(x)
        
        skip_connections = []
        for down in self.downs:
            x = down(x, t)
            skip_connections.append(x)
            x = self.pool(x)
            
        x = self.mid(x, t)
        
        for up in self.ups:
            x = self.up_sample(x)
            skip_x = skip_connections.pop()
            if x.shape[-1] != skip_x.shape[-1]:
                x = F.pad(x, (0, skip_x.shape[-1] - x.shape[-1]))
            x = torch.cat((x, skip_x), dim=1)
            x = up(x, t)
            
        return self.final_conv(x)

def linear_beta_schedule(timesteps):
    beta_start = 0.0001
    beta_end = 0.02
    return torch.linspace(beta_start, beta_end, timesteps)

class GaussianDiffusion1D:
    def __init__(self, model, timesteps=500):
        self.model = model
        self.timesteps = timesteps
        self.betas = linear_beta_schedule(timesteps)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        
    def sample_timesteps(self, n, device):
        return torch.randint(low=1, high=self.timesteps, size=(n,), device=device)
        
    def add_noise(self, x_start, t):
        noise = torch.randn_like(x_start)
        alphas_cumprod_t = self.alphas_cumprod.to(x_start.device)[t].view(-1, 1, 1)
        x_noisy = torch.sqrt(alphas_cumprod_t) * x_start + torch.sqrt(1 - alphas_cumprod_t) * noise
        return x_noisy, noise

    def compute_loss(self, x_start, cond):
        device = x_start.device
        t = self.sample_timesteps(x_start.shape[0], device)
        x_noisy, noise = self.add_noise(x_start, t)
        predicted_noise = self.model(x_noisy, t, cond)
        return F.l1_loss(predicted_noise, noise)

    @torch.no_grad()
    def sample(self, cond):
        self.model.eval()
        device = cond.device
        b, c, l = cond.shape
        x = torch.randn((b, c, l), device=device)
        
        for i in reversed(range(1, self.timesteps)):
            t = (torch.ones(b) * i).long().to(device)
            predicted_noise = self.model(x, t, cond)
            
            alpha = self.alphas[i].to(device)
            alpha_cumprod = self.alphas_cumprod[i].to(device)
            beta = self.betas[i].to(device)
            
            if i > 1:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)
                
            x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / (torch.sqrt(1 - alpha_cumprod))) * predicted_noise) + torch.sqrt(beta) * noise
            
        self.model.train()
        return x

In [ ]:
def calculate_exhaustive_metrics(clean, noisy, denoised):
    c = clean.cpu().numpy().squeeze()
    n = noisy.cpu().numpy().squeeze()
    d = denoised.cpu().numpy().squeeze()
    if len(c.shape) == 1:
        c, n, d = c[None, :], n[None, :], d[None, :]
        
    metrics = {}
    
    # 1. Cosine Similarity
    cos_sim = np.sum(c * d, axis=-1) / (np.linalg.norm(c, axis=-1) * np.linalg.norm(d, axis=-1) + 1e-8)
    metrics['Cosine_Sim'] = np.mean(cos_sim)
    
    # 2. Pearson R
    c_mean = np.mean(c, axis=-1, keepdims=True)
    d_mean = np.mean(d, axis=-1, keepdims=True)
    num = np.sum((c - c_mean) * (d - d_mean), axis=-1)
    den = np.sqrt(np.sum((c - c_mean)**2, axis=-1) * np.sum((d - d_mean)**2, axis=-1)) + 1e-8
    metrics['Pearson_r'] = np.mean(num / den)
    
    # 3. RMSE
    rmse = np.sqrt(np.mean((c - d)**2, axis=-1))
    metrics['RMSE'] = np.mean(rmse)
    
    # 4. MAE
    mae = np.mean(np.abs(c - d), axis=-1)
    metrics['MAE'] = np.mean(mae)
    
    # 5. SSD (DeScoD-ECG)
    ssd = np.sum((c - d)**2, axis=-1)
    metrics['SSD'] = np.mean(ssd)
    
    # 6. MAD (Maximum Absolute Distance)
    mad = np.max(np.abs(c - d), axis=-1)
    metrics['MAD'] = np.mean(mad)
    
    # 7. PRD
    prd = np.sqrt(np.sum((c - d)**2, axis=-1) / (np.sum(c**2, axis=-1) + 1e-8)) * 100
    metrics['PRD'] = np.mean(prd)
    
    # 8. SNR
    signal_power = np.mean(c**2, axis=-1)
    noise_power = np.mean((c - d)**2, axis=-1) + 1e-8
    snr = 10 * np.log10(signal_power / noise_power)
    metrics['SNR'] = np.mean(snr)
    
    # 9. SNR_med (FIXED case sensitivity)
    noise_power_med = np.median((c - d)**2, axis=-1) + 1e-8
    snr_med = 10 * np.log10(signal_power / noise_power_med)
    metrics['SNR_med'] = np.mean(snr_med)
    
    # 10. Improved SNR (TFCDiff)
    noisy_noise_power = np.mean((c - n)**2, axis=-1) + 1e-8
    noisy_snr = 10 * np.log10(signal_power / noisy_noise_power)
    metrics['ImSNR'] = np.mean(snr - noisy_snr)
    
    # 11. KS Statistic (PTB-IMAGE)
    ks_vals = [ks_2samp(c[i], d[i]).statistic for i in range(c.shape[0])]
    metrics['KS_Stat'] = np.mean(ks_vals)
    
    # 12. WAD (Weighted Absolute Difference)
    ranges = (np.max(c, axis=-1) - np.min(c, axis=-1)) + 1e-8
    wad = np.mean(np.abs(c - d), axis=-1) / ranges
    metrics['WAD'] = np.mean(wad)

    return metrics

## Training Phase (Scrollable Log)

In [ ]:
# Global variables so we can plot them in the next cell
history = {k: [] for k in ['epoch', 'loss', 'Cosine_Sim', 'Pearson_r', 'RMSE', 'MAE', 'SSD', 'MAD', 'PRD', 'SNR', 'SNR_med', 'ImSNR', 'KS_Stat', 'WAD']}
final_clean, final_cond, final_denoised = None, None, None

def train():
    global final_clean, final_cond, final_denoised, history
    try:
        dataset = RealECGDataset(PATH_CLEAN, PATH_NOISY)
    except FileNotFoundError as e:
        print(e)
        return

    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    model = ConditionalUNet1D(channels=1).to(device)
    diffusion = GaussianDiffusion1D(model, timesteps=500)
    
    optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
    epochs = 50
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()

    best_cos_sim = -1.0
    
    print("\nStarting Training Loop...\n")
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        
        for batch_idx, (clean, cond) in enumerate(dataloader):
            clean = clean.to(device)
            cond = cond.to(device)
            
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                loss = diffusion.compute_loss(clean, cond)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            
        scheduler.step()
        avg_loss = total_loss / len(dataloader)
        
        # Validation Step
        model.eval()
        with torch.no_grad():
            # Take 4 samples for quick validation
            val_cond = cond[:4]
            val_clean = clean[:4]
            denoised = diffusion.sample(val_cond)
            
            metrics = calculate_exhaustive_metrics(val_clean, val_cond, denoised)
            
            # Save for plotting later
            if epoch == epochs:
                final_clean, final_cond, final_denoised = val_clean, val_cond, denoised
            
            history['epoch'].append(epoch)
            history['loss'].append(avg_loss)
            for k in metrics.keys():
                history[k].append(metrics[k])
            
            # Scrollable Print Output
            save_msg = ""
            if metrics['Cosine_Sim'] > best_cos_sim:
                best_cos_sim = metrics['Cosine_Sim']
                torch.save(model.state_dict(), 'best_diffusion_model.pth')
                save_msg = "(Saved Best Model)"
                
            print(f"Epoch {epoch:02d}/{epochs} | Loss: {avg_loss:.4f} | CosSim: {metrics['Cosine_Sim']:.4f} | SNR_med: {metrics['SNR_med']:.2f} dB | MAD: {metrics['MAD']:.4f} {save_msg}")

    pd.DataFrame(history).to_csv("exhaustive_metrics_history.csv", index=False)
    print("\nTraining Complete! All 12 metrics saved to 'exhaustive_metrics_history.csv'")
    
    # Pass back variables for the next cells
    return model, diffusion, dataloader

model, diffusion, dataloader = train()

## Post-Training Visualizations
Run this cell AFTER training to plot the learning curves and thesis artifacts.

In [ ]:
# 1. Plot Learning Curves
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
ax1.plot(history['epoch'], history['loss'], color='purple', marker='o')
ax1.set_title("Training Loss (L1)", fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.grid(True, alpha=0.3)

ax2.plot(history['epoch'], history['SNR_med'], color='orange', marker='o')
ax2.set_title("Median SNR Progression (dB)", fontweight='bold')
ax2.set_xlabel("Epoch")
ax2.grid(True, alpha=0.3)

ax3.plot(history['epoch'], history['Cosine_Sim'], color='green', marker='o')
ax3.set_title("Cosine Similarity Progression", fontweight='bold')
ax3.set_xlabel("Epoch")
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("thesis_learning_curves.png", dpi=300)
plt.show()

# 2. Plot Signal Artifacts
c = final_clean[0, 0].cpu().numpy()
n = final_cond[0, 0].cpu().numpy()
d = final_denoised[0, 0].cpu().numpy()

fig2 = plt.figure(figsize=(18, 12))
# Overlay
ax4 = plt.subplot(3, 1, 1)
ax4.plot(c, label="Ground Truth (Clean)", color="#2ca02c", alpha=0.8, linewidth=2)
ax4.plot(n, label="Digitizer Output (Noisy)", color="#d62728", alpha=0.5, linestyle="--")
ax4.plot(d, label="Diffusion Output (Denoised)", color="#1f77b4", alpha=0.9, linewidth=1.5)
ax4.set_title("Final Epoch - Full ECG Reconstruction", fontsize=14, fontweight='bold')
ax4.legend(loc="upper right")
ax4.grid(True, alpha=0.3)

# Zoom
peak_idx = np.argmax(c[:800]) 
start, end = max(0, peak_idx - 50), min(len(c), peak_idx + 80)
ax5 = plt.subplot(3, 2, 3)
ax5.step(range(start, end), n[start:end], label="Noisy Extraction (Staircase)", color="#d62728", alpha=0.6, where='mid')
ax5.plot(range(start, end), c[start:end], label="Clean Signal", color="#2ca02c", linewidth=2)
ax5.plot(range(start, end), d[start:end], label="Diffusion Denoised", color="#1f77b4", linewidth=2)
ax5.set_title("Zoomed QRS Complex (Staircase Smoothing)", fontsize=12, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Error
ax6 = plt.subplot(3, 2, 4)
ax6.plot(np.abs(c - n), label="Digitizer Error |Clean - Noisy|", color="#d62728", alpha=0.5)
ax6.plot(np.abs(c - d), label="Diffusion Error |Clean - Denoised|", color="#1f77b4", alpha=0.8)
ax6.set_title("Absolute Error Distribution", fontsize=12, fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)

# PSD
freqs_c, psd_c = welch(c, fs=500, nperseg=256)
freqs_n, psd_n = welch(n, fs=500, nperseg=256)
freqs_d, psd_d = welch(d, fs=500, nperseg=256)
ax7 = plt.subplot(3, 1, 3)
ax7.semilogy(freqs_n, psd_n, label="Noisy PSD (Digitizer Artifacts)", color="#d62728", alpha=0.6)
ax7.semilogy(freqs_c, psd_c, label="Clean PSD (Ground Truth)", color="#2ca02c", linewidth=2)
ax7.semilogy(freqs_d, psd_d, label="Denoised PSD (Diffusion Output)", color="#1f77b4", linewidth=2)
ax7.set_title("Power Spectral Density (High-Frequency Noise Removal)", fontsize=14, fontweight='bold')
ax7.set_xlabel("Frequency (Hz)")
ax7.set_ylabel("Power/Frequency (dB/Hz)")
ax7.legend()
ax7.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("thesis_final_artifacts.png", dpi=300)
plt.show()

## Thesis Comparison Table (Baseline vs Model)
Run this to get the exact numbers to copy and paste into your Results section.

In [ ]:
model.eval()
with torch.no_grad():
    # Get one batch for final evaluation
    clean, noisy = next(iter(dataloader))
    clean, noisy = clean.to(device), noisy.to(device)
    
    # 1. BASELINE METRICS (Comparing Clean to Raw Noisy Digitizer)
    baseline_metrics = calculate_exhaustive_metrics(clean, noisy, noisy)
    
    # 2. MODEL METRICS (Comparing Clean to Denoised)
    denoised = diffusion.sample(noisy)
    model_metrics = calculate_exhaustive_metrics(clean, noisy, denoised)

print("===============================================================")
print("          FINAL THESIS COMPARISON: BASELINE VS MODEL           ")
print("===============================================================")
print(f"{'Metric':<12} | {'Baseline (Digitizer Only)':<25} | {'Your Diffusion Model'}")
print("-" * 63)

for k in baseline_metrics.keys():
    if k == 'ImSNR': 
        continue # Baseline cannot have an ImSNR against itself
    print(f"{k:12} | {baseline_metrics[k]:<25.4f} | {model_metrics[k]:.4f}")

print("\n(Note: ImSNR of the model was {:.4f} dB)".format(model_metrics['ImSNR']))

## 📖 How to Decipher These Results for your Thesis

When you write your results section, you need to explain **why** these numbers matter. Here is your decoder ring:

### 1. The "Shape" Metrics (Cosine_Sim & Pearson_r)
* **What they mean:** They measure if the biological waveform (the shape of the heartbeat) was preserved.
* **How to compare:** In the **DeScoD-ECG (2024)** paper, they achieved a Cosine Similarity of ~0.92 when removing hospital noise. If your model gets `> 0.95`, you write: *"Our conditional architecture achieves superior morphological preservation (CosSim = X) compared to prior score-based models (CosSim = 0.92), ensuring no diagnostic features were hallucinated."*

### 2. The "Distance & Spike" Metrics (MAD, RMSE, SSD)
* **What they mean:** `MAD` (Maximum Absolute Distance) finds the single largest spike in the signal. Optical digitizers create huge spikes when they misread text/grids.
* **How to compare:** Look at the Table above. The Baseline `MAD` will be high. Your Model `MAD` will be low. You write: *"The diffusion refinement successfully suppressed catastrophic outlier spikes, reducing the Maximum Absolute Distance from [Baseline MAD] to [Model MAD]."*

### 3. The "Noise" Metrics (SNR_med, PRD)
* **What they mean:** `SNR_med` ignores isolated spikes and measures overall noise. `PRD` is the standard for ECG compression.
* **How to compare:** In the **ECG-Image-Kit (2024)** paper, they got an $SNR_{med}$ of 26.54 dB on *synthetic, perfect* images. If your model gets anywhere near 25-27 dB on *real scanned* images, you write: *"Despite being evaluated on heavily distorted, real-world scanned records (unlike the synthetic benchmarks in Shivashankara et al.), our model achieved a highly competitive Median SNR of X dB."*

### 4. The "Distribution" Metrics (KS_Stat)
* **What they mean:** The Kolmogorov-Smirnov distance checks if the probability distribution of your output matches real biology.
* **How to compare:** Lower is better. The **PTB-IMAGE (2025)** paper used this. You write: *"A KS Statistic of [Model KS] demonstrates that the generative output mathematically aligns with the true biological distribution, correcting the statistical drift introduced by optical quantization."*